# Week 13 실습 · matplotlib 데이터 시각화

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MNU-AI-Programming/AI-Assisted-Python-Programming/blob/main/weeks/week13/practice.ipynb)

**AI활용코딩연습(ZBE135)** · 이영호 교수 · 국립목포대학교 컴퓨터학부

## 오늘의 목표

1. `fig, ax = plt.subplots()`로 시작하는 **객체지향 방식**으로 그래프를 그립니다.
2. **질문에 맞는 차트**(추세·비교·분포·관계·구성)를 고릅니다.
3. 제목에 결론, 축에 단위, 강조 색, 주석으로 **메시지가 분명한 차트**를 만듭니다.
4. 나쁜 차트(잘린 축, 정렬 안 됨, 과한 색)를 고치고, 여러 차트를 한 장의 **대시보드**로 저장합니다.
5. AI가 만든 차트 코드를 **체크리스트와 자동 점검 함수**로 검증합니다.

| 구간 | 내용 | 시간 |
|---|---|---|
| 1 | Level 1 matplotlib 기본 구조 | 15분 |
| 2 | Level 1 질문별 차트 6종 | 25분 |
| 3 | 메시지가 분명한 차트 | 15분 |
| 4 | 나쁜 차트 고치기 · Level 2 바꾸기 | 15분 |
| 5 | Level 3 카페 대시보드 | 25분 |
| 6 | Level 4 AI 차트 코드 점검 | 15분 |
| 7 | 정리 | 10분 |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from pathlib import Path
import urllib.request

try:
    import koreanize_matplotlib            # 그래프 한글 폰트
except ModuleNotFoundError:
    %pip install -q koreanize-matplotlib
    import koreanize_matplotlib

DATA_URL = "https://raw.githubusercontent.com/MNU-AI-Programming/AI-Assisted-Python-Programming/main/datasets/"

def get_data(name):
    for folder in [Path("../../datasets"), Path("datasets"), Path(".")]:
        if (folder / name).exists():
            return str(folder / name)
    urllib.request.urlretrieve(DATA_URL + name, name)
    return name

# 12주차에서 만든 분석용 데이터 준비
cafe = pd.read_csv(get_data("cafe_sales.csv"), parse_dates=["date"])
weather = pd.read_csv(get_data("weather_daily.csv"), parse_dates=["date"])
cafe["revenue"] = cafe["unit_price"] * cafe["qty"]
cafe["hour"] = cafe["time"].str[:2].astype(int)

daily = cafe.groupby("date").agg(revenue=("revenue", "sum"), orders=("order_id", "size")).reset_index()
daily["weekday"] = daily["date"].dt.dayofweek.map(dict(enumerate("월화수목금토일")))
drinks = cafe[cafe["temp_type"].isin(["HOT", "ICE"])]
ice = drinks.pivot_table(index="date", columns="temp_type", values="qty", aggfunc="sum", fill_value=0)
daily = daily.merge((ice["ICE"] / ice.sum(axis=1)).rename("ice_ratio").reset_index(), on="date")
daily = daily.merge(weather[["date", "avg_temp"]], on="date")
menu = cafe.groupby("menu")["revenue"].sum().sort_values()
print(daily.shape, menu.shape)

---
## 1. Level 1 따라하기: matplotlib 기본 구조

```text
Figure (그림 전체, 종이)
 └── Axes (그래프 한 칸, 좌표평면)  ← 대부분의 작업은 ax.메서드()
      ├── 제목 ax.set_title
      ├── 축 레이블 ax.set_xlabel / ax.set_ylabel
      ├── 그래프 요소 ax.plot / ax.bar / ax.scatter ...
      └── 범례 ax.legend
```

`plt.plot(...)`처럼 바로 그리는 방식도 있지만, 여러 그래프를 다루려면 **`fig, ax = plt.subplots()`로 시작하는 객체지향 방식**이 명확합니다.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3))
ax.plot([1, 2, 3, 4], [10, 30, 20, 40], marker="o", label="예시 값")
ax.set_title("제목")
ax.set_xlabel("x축 이름 (단위)")
ax.set_ylabel("y축 이름 (단위)")
ax.legend()
plt.show()

---
## 2. Level 1 따라하기: 질문별 차트 6종

| 질문 | 차트 | 메서드 |
|---|---|---|
| 시간에 따라 어떻게 변하나? (추세) | 선 그래프 | `ax.plot` |
| 무엇이 더 큰가? (비교) | 막대 그래프 (항목 많으면 가로·정렬) | `ax.bar`, `ax.barh` |
| 값이 어떻게 퍼져 있나? (분포) | 히스토그램 | `ax.hist` |
| 그룹별 분포 비교 | 상자 그림 | `ax.boxplot` |
| 두 변수는 관계가 있나? (관계) | 산점도 | `ax.scatter` |
| 전체에서 차지하는 비율 (구성) | 정렬된 막대 / 누적 막대 (파이는 2~3개 항목일 때만) | `ax.barh`, `bottom=` |

### ① 추세: 일 매출과 7일 이동평균

In [ ]:
daily["ma7"] = daily["revenue"].rolling(7).mean()

fig, ax = plt.subplots(figsize=(9, 3.5))
ax.plot(daily["date"], daily["revenue"] / 1000, color="#BBBBBB", linewidth=1, label="일 매출")
ax.plot(daily["date"], daily["ma7"] / 1000, color="#1f4e79", linewidth=2.5, label="7일 이동평균")
ax.set_title("일 매출 추이: 4월 말과 6월 중순(시험 주간)에 높아짐")
ax.set_ylabel("매출 (천 원)")
ax.legend(loc="upper left")
plt.show()

### ② 비교: 요일별 일평균 매출 (막대)

In [ ]:
order = list("월화수목금토")
by_wd = daily.groupby("weekday")["revenue"].mean().reindex(order) / 1000

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.bar(by_wd.index, by_wd.values, color="#4C78A8")
ax.set_title("요일별 일평균 매출")
ax.set_ylabel("일평균 매출 (천 원)")
plt.show()

### ③ 비교(항목이 많을 때): 메뉴별 매출 – **가로 막대 + 정렬**

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.barh(menu.index, menu.values / 10000, color="#4C78A8")
ax.set_title("메뉴별 매출")
ax.set_xlabel("매출 (만 원)")
plt.show()

### ④ 분포: 주문 시각 히스토그램

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.hist(cafe["hour"], bins=range(8, 21), edgecolor="white", color="#72B7B2")
ax.set_title("주문이 몰리는 시각: 12~13시")
ax.set_xlabel("주문 시각 (시)")
ax.set_ylabel("주문 건수")
ax.set_xticks(range(8, 20))
plt.show()

### ⑤ 그룹별 분포: 요일별 일 매출 상자 그림

상자 = 가운데 50%(Q1~Q3), 가운데 선 = 중앙값, 수염 = 일반적인 범위, 점 = 이상값 후보

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
data_by_day = [daily.loc[daily["weekday"] == d, "revenue"] / 1000 for d in order]
try:
    ax.boxplot(data_by_day, tick_labels=order)    # matplotlib 3.9 이상
except TypeError:
    ax.boxplot(data_by_day, labels=order)         # 이전 버전
ax.set_title("요일별 일 매출 분포")
ax.set_ylabel("일 매출 (천 원)")
plt.show()

### ⑥ 관계: 기온과 아이스 음료 비율 (산점도)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(daily["avg_temp"], daily["ice_ratio"] * 100, s=18, alpha=0.7, color="#E45756")
ax.set_title("기온이 높을수록 아이스 음료 비율이 높다")
ax.set_xlabel("평균기온 (℃)")
ax.set_ylabel("아이스 비율 (%)")

slope, intercept = np.polyfit(daily["avg_temp"], daily["ice_ratio"] * 100, 1)   # 추세선
xs = np.linspace(daily["avg_temp"].min(), daily["avg_temp"].max(), 50)
ax.plot(xs, slope * xs + intercept, color="#333333", linestyle="--", linewidth=1)
plt.show()
print(f"기온 1℃ 오를 때 아이스 비율 약 {slope:.1f}%p 증가 (상관이지 인과는 아님)")

---
## 3. 메시지가 분명한 차트

같은 데이터라도 **무엇을 말하고 싶은지**가 드러나게 그립니다.

| 원칙 | 방법 |
|---|---|
| 제목 = 결론 | "메뉴별 매출" → "아메리카노가 매출의 4분의 1" |
| 축에 단위 | "매출 (만 원)", "평균기온 (℃)" |
| 하나만 강조 | 강조할 막대만 진한 색, 나머지는 회색 |
| 숫자 직접 표시 | `ax.bar_label`, `ax.annotate` |
| 군더더기 제거 | 위·오른쪽 테두리 제거, 불필요한 격자·범례 제거 |

In [ ]:
share = menu / menu.sum() * 100
colors = ["#BBBBBB"] * (len(share) - 1) + ["#C0392B"]      # 마지막(가장 큰 값)만 강조

fig, ax = plt.subplots(figsize=(6.5, 4.2))
bars = ax.barh(share.index, share.values, color=colors)
ax.bar_label(bars, fmt="%.1f%%", padding=3, fontsize=9)
ax.set_title("아메리카노가 매출의 약 4분의 1을 차지한다", loc="left", fontweight="bold")
ax.set_xlabel("매출 비중 (%)")
ax.spines[["top", "right"]].set_visible(False)
ax.set_xlim(0, share.max() * 1.15)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.5))
ax.plot(daily["date"], daily["ma7"] / 1000, color="#1f4e79", linewidth=2.5)
for start, end, label in [("2025-04-21", "2025-04-25", "중간고사"), ("2025-06-16", "2025-06-20", "기말고사")]:
    s, e = pd.Timestamp(start), pd.Timestamp(end)
    ax.axvspan(mdates.date2num(s), mdates.date2num(e), color="#F4D03F", alpha=0.3)
    ax.annotate(label, xy=(mdates.date2num(s), daily["ma7"].max() / 1000), xytext=(5, -5),
                textcoords="offset points", fontsize=9)
ax.set_title("시험 주간에 일 매출(7일 이동평균)이 올라간다", loc="left", fontweight="bold")
ax.set_ylabel("매출 (천 원)")
ax.spines[["top", "right"]].set_visible(False)
plt.show()

---
## 4. 나쁜 차트 고치기 · Level 2 바꾸기

### ❌ 나쁜 예: 무엇이 문제일까요?

In [ ]:
wd_mean = daily.groupby("weekday")["revenue"].mean()          # 가나다순(금목수월토화)으로 정렬됨

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.bar(wd_mean.index, wd_mean.values, color=plt.cm.rainbow(np.linspace(0, 1, 6)))
ax.set_ylim(60000, 175000)
ax.set_title("data")
plt.show()

| 문제 | 왜 나쁜가 | 고치기 |
|---|---|---|
| y축이 60,000에서 시작 | 막대 길이가 실제 비율을 왜곡 (토요일이 거의 0처럼 보임) | **막대 그래프는 0부터** |
| 요일 순서가 가나다순 | 읽는 사람이 순서를 다시 맞춰야 함 | `reindex(["월", …])` |
| 무지개 색 | 색이 의미 없이 주의를 분산 | 한 색 + 필요하면 강조 하나 |
| 제목 "data", 축 레이블 없음 | 무엇을 보여 주는지 모름 | 결론형 제목, 단위 |

### ✏️ 직접 해보기

아래 셀을 고쳐서 위 네 가지 문제를 모두 해결하세요. 토요일만 다른 색으로 강조해 "토요일 매출은 평일의 절반 이하"라는 메시지를 만들어 보세요.

In [ ]:
wd_mean = daily.groupby("weekday")["revenue"].mean()          # ← 순서를 고쳐 보세요

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.bar(wd_mean.index, wd_mean.values / 1000, color="#4C78A8")  # ← 색을 고쳐 보세요
ax.set_title("data")                                            # ← 제목과 축 레이블을 고쳐 보세요
plt.show()

---
## 5. Level 3 적용하기: 카페 대시보드

여러 차트를 한 장에 모으면 **전체 그림**을 한눈에 볼 수 있습니다. `plt.subplots(2, 2)`는 2×2 격자의 Axes 배열을 돌려줍니다.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
fig.suptitle("캠퍼스 카페 2025년 1학기 판매 요약 (가상 데이터)", fontsize=15, fontweight="bold")

# (0,0) 추세
ax = axes[0, 0]
ax.plot(daily["date"], daily["revenue"] / 1000, color="#CCCCCC", linewidth=1)
ax.plot(daily["date"], daily["ma7"] / 1000, color="#1f4e79", linewidth=2)
ax.set_title("일 매출과 7일 이동평균", loc="left")
ax.set_ylabel("천 원")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%m/%d"))     # 칸이 좁으면 날짜 글자가 겹친다 → 짧은 형식

# (0,1) 메뉴 비교
ax = axes[0, 1]
ax.barh(share.index, share.values, color=colors)
ax.set_title("메뉴별 매출 비중", loc="left")
ax.set_xlabel("%")

# (1,0) 요일 × 시간대
cafe["daypart"] = pd.cut(cafe["hour"], [7, 10, 13, 16, 19], labels=["아침", "점심", "오후", "저녁"])
cafe["weekday"] = cafe["date"].dt.dayofweek.map(dict(enumerate("월화수목금토일")))
days_per_wd = daily.groupby("weekday").size()
heat = (cafe.pivot_table(index="weekday", columns="daypart", values="revenue", aggfunc="sum", observed=True)
            .reindex(order).div(days_per_wd.reindex(order), axis=0) / 1000)
ax = axes[1, 0]
im = ax.imshow(heat.values, cmap="Blues", aspect="auto")
ax.set_xticks(range(heat.shape[1]), heat.columns)
ax.set_yticks(range(heat.shape[0]), heat.index)
for i in range(heat.shape[0]):
    for j in range(heat.shape[1]):
        v = heat.values[i, j]
        ax.text(j, i, f"{v:.0f}", ha="center", va="center", color="white" if v > 50 else "black", fontsize=9)
ax.set_title("요일 × 시간대 일평균 매출 (천 원)", loc="left")
fig.colorbar(im, ax=ax, shrink=0.8)

# (1,1) 관계
ax = axes[1, 1]
ax.scatter(daily["avg_temp"], daily["ice_ratio"] * 100, s=14, alpha=0.7, color="#E45756")
ax.set_title("기온과 아이스 음료 비율", loc="left")
ax.set_xlabel("평균기온 (℃)")
ax.set_ylabel("아이스 비율 (%)")

for a in axes.flat:
    a.spines[["top", "right"]].set_visible(False)
fig.tight_layout()

Path("figures").mkdir(exist_ok=True)
fig.savefig("figures/cafe_dashboard.png", dpi=150, bbox_inches="tight")
plt.show()
print("저장:", Path("figures/cafe_dashboard.png").stat().st_size // 1024, "KB")

> 💡 `savefig`는 `plt.show()`보다 **먼저** 호출합니다. (`show` 후에는 그림이 비워질 수 있습니다.)
> 보고서·발표용은 `dpi=150` 이상, 여백은 `bbox_inches="tight"`.

---
## 6. Level 4 AI와 함께: 차트 코드 점검

"요일별 매출을 막대 그래프로 그려 줘"라고 요청하면 AI는 대개 **동작하는** 코드를 줍니다. 하지만 위의 나쁜 예처럼 **보기 좋은지, 정확한지**는 따로 확인해야 합니다.

눈으로만 보지 말고 **그래프 객체를 코드로 점검**할 수 있습니다.

In [ ]:
def audit_chart(ax, kind="bar"):
    """그래프 품질 체크리스트를 자동으로 점검해 문제 목록을 돌려준다."""
    problems = []
    title = " ".join(ax.get_title(loc=loc) for loc in ("left", "center", "right")).strip()
    if not title or title.lower() in {"data", "chart", "plot"}:
        problems.append("제목이 없거나 의미 없음")
    if not ax.get_xlabel() and not ax.get_ylabel():
        problems.append("축 레이블 없음")
    if kind == "bar" and ax.get_ylim()[0] > 0:
        problems.append("막대 그래프의 y축이 0에서 시작하지 않음")
    if kind == "barh" and ax.get_xlim()[0] > 0:
        problems.append("가로 막대 그래프의 x축이 0에서 시작하지 않음")
    colors = {tuple(p.get_facecolor()) for p in ax.patches}
    if len(colors) > 3:
        problems.append(f"색이 너무 많음 ({len(colors)}가지)")
    if len(ax.lines) > 1 and ax.get_legend() is None:
        problems.append("선이 여러 개인데 범례 없음")
    return problems


# AI가 준 코드 (나쁜 예와 같은 그래프)
fig, ax = plt.subplots(figsize=(5, 3))
ax.bar(wd_mean.index, wd_mean.values, color=plt.cm.rainbow(np.linspace(0, 1, 6)))
ax.set_ylim(60000, 175000)
ax.set_title("data")
plt.close(fig)
print("AI 차트 점검:", audit_chart(ax))

In [ ]:
# 고친 차트
wd_fixed = daily.groupby("weekday")["revenue"].mean().reindex(order) / 1000
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.bar(wd_fixed.index, wd_fixed.values, color=["#BBBBBB"] * 5 + ["#C0392B"])
ax.set_title("토요일 일평균 매출은 평일의 절반 이하", loc="left", fontweight="bold")
ax.set_ylabel("일평균 매출 (천 원)")
ax.spines[["top", "right"]].set_visible(False)
print("고친 차트 점검:", audit_chart(ax) or "문제 없음 ✅")
plt.show()

### 📝 AI에게 차트를 요청할 때

```text
daily DataFrame(열: date, revenue(원), weekday('월'~'토'))으로 요일별 일평균 매출 막대 그래프를 그려 주세요.
- fig, ax = plt.subplots() 객체지향 방식
- x축은 월~토 순서, y축은 천 원 단위, 0부터 시작
- 제목은 그래프의 결론을 문장으로, 축에 단위 표시
- 토요일만 빨간색, 나머지는 회색
- 한글 폰트는 koreanize_matplotlib 사용
```

| 체크리스트 | |
|---|---|
| 질문에 맞는 차트 종류인가? | 추세·비교·분포·관계·구성 |
| 제목이 결론을 말하나? 축에 단위가 있나? | |
| 막대는 0부터 시작하나? 순서가 의미 있나? | |
| 색은 의미가 있을 때만 쓰였나? | |
| 데이터와 그래프 숫자가 일치하나? | 표의 값과 대조 |

---
## 7. 정리

| 오늘 배운 것 | 핵심 |
|---|---|
| 구조 | Figure(종이) · Axes(그래프 한 칸), `fig, ax = plt.subplots()` |
| 차트 선택 | 추세→선, 비교→막대(정렬), 분포→히스토그램·상자, 관계→산점도 |
| 메시지 | 결론형 제목, 단위, 하나만 강조, 값 표시, 군더더기 제거 |
| 나쁜 차트 | 잘린 축, 무의미한 순서·색, 레이블 없음 |
| 대시보드 | `plt.subplots(2, 2)`, `suptitle`, `tight_layout`, `savefig(dpi=150)` |
| AI 검증 | 체크리스트 + 그래프 객체 자동 점검(`ax.get_title()`, `ax.patches`, `ax.get_ylim()`) |

### 자기 점검

- [ ] 질문 다섯 종류에 맞는 차트를 고를 수 있다.
- [ ] 막대 그래프가 0부터 시작해야 하는 이유를 설명할 수 있다.
- [ ] 결론형 제목과 단위가 있는 차트를 만들 수 있다.
- [ ] `savefig`를 `show`보다 먼저 호출하는 이유를 안다.

### 다음 주 예고

**Week 14 · AI 디버깅 · 테스트 · 리팩토링**: 결함이 있는 분석 코드를 진단하고, 테스트를 붙이고, 읽기 좋게 고칩니다. 최종 프로젝트 노트북도 이 방법으로 점검합니다.